In [ ]:
import warnings
from glob import glob

import pandas as pd
import seaborn as sns
from category_encoders import OneHotEncoder
from IPython.display import VimeoVideo
from ipywidgets import Dropdown, FloatSlider, IntSlider, interact
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge  # noqa F401
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:
VimeoVideo("656842813", h="07f074324e", width=600)


In [ ]:
def wrangle(filepath):
    # Read CSV file
    df = pd.read_csv(filepath)

    # Subset data: Apartments in "Capital Federal", less than 400,000
    mask_ba = df["place_with_parent_names"].str.contains("Capital Federal")
    mask_apt = df["property_type"] == "apartment"
    mask_price = df["price_aprox_usd"] < 400_000
    df = df[mask_ba & mask_apt & mask_price]

    # Subset data: Remove outliers for "surface_covered_in_m2"
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low, high)
    df = df[mask_area]

    # Split "lat-lon" column
    df[["lat", "lon"]] = df["lat-lon"].str.split(",", expand=True).astype(float)
    df.drop(columns="lat-lon", inplace=True)

    # Get place name
    df["neighborhood"] = df["place_with_parent_names"].str.split("|", expand=True)[3]
    df.drop(columns="place_with_parent_names", inplace=True)

    # Drop features with high null values
    df.drop(columns=["floor","expenses"], inplace=True)

    # Drop low-High cardinality categorical featrues 
    df.drop(columns=["operation","property_type","currency","properati_url"], inplace=True)

    # Drop liky columns
    df.drop(columns=[
        'price',
        'price_aprox_local_currency',
        'price_per_m2',
        'price_usd_per_m2'], inplace=True
    )
    # Drop columns with multicollinearity 
    df.drop(columns=["surface_total_in_m2","rooms"], inplace=True)
   
    return df


In [ ]:
VimeoVideo("656842538", h="bd85634eb1", width=600)


In [ ]:
files = glob("data/buenos-aires-real-estate-*.csv")
files


In [ ]:
# Check your work
assert len(files) == 5, f"`files` should contain 5 items, not {len(files)}"


In [ ]:
VimeoVideo("656842076", h="0f654d427f", width=600)


In [ ]:
# [expression for item in iterable if condition]
frames = [wrangle(file) for file in files ]


In [ ]:
# Check your work
assert len(frames) == 5, f"`frames` should contain 5 items, not {len(frames)}"
assert all(
    [isinstance(frame, pd.DataFrame) for frame in frames]
), "The items in `frames` should all be DataFrames."


In [ ]:
VimeoVideo("656841910", h="79c7dbc5ab", width=600)


In [ ]:
df = pd.concat([wrangle(file) for file in files ], ignore_index=True)
print(df.info())
df.head()


In [ ]:
# Check your work
assert len(df) == 6582, f"`df` has the wrong number of rows: {len(df)}"
assert df.shape[1] <= 17, f"`df` has too many columns: {df.shape[1]}"


In [ ]:
VimeoVideo("656848648", h="6964fa0c8c", width=600)


In [ ]:
# Check your work
assert len(df) == 6582, f"`df` has the wrong number of rows: {len(df)}"
assert df.shape[1] <= 15, f"`df` has too many columns: {df.shape[1]}"


In [ ]:
VimeoVideo("656848196", h="37dbc44b09", width=600)


In [ ]:
df.select_dtypes("object").nunique()


In [ ]:
# Check your work
assert len(df) == 6582, f"`df` has the wrong number of rows: {len(df)}"
assert df.shape[1] <= 11, f"`df` has too many columns: {df.shape[1]}"


In [ ]:
VimeoVideo("656847896", h="11de775937", width=600)


In [ ]:
sorted(df.columns)


In [ ]:
# Check your work
assert len(df) == 6582, f"`df` has the wrong number of rows: {len(df)}"
assert df.shape[1] <= 7, f"`df` has too many columns: {df.shape[1]}"


In [ ]:
VimeoVideo("656847237", h="4b5cfed5d6", width=600)


In [ ]:
corr = df.select_dtypes("number").drop(columns=["price_aprox_usd"]).corr()
sns.heatmap(corr)


In [ ]:
# Check your work
assert len(df) == 6582, f"`df` has the wrong number of rows: {len(df)}"
assert df.shape[1] == 5, f"`df` has the wrong number of columns: {df.shape[1]}"
df.head()


In [ ]:
target = "price_aprox_usd"
y_train = df[target]
feature_matrix = ["surface_covered_in_m2","lat","lon","neighborhood"]
X_train = df[feature_matrix]


In [ ]:
# Check your work
assert X_train.shape == (6582, 4), f"`X_train` is the wrong size: {X_train.shape}."
assert y_train.shape == (6582,), f"`y_train` is the wrong size: {y_train.shape}."


In [ ]:
VimeoVideo("656849559", h="bca444c8af", width=600)


In [ ]:
# Mean of target
y_mean = y_train.mean()

# Baseline predictions: predict mean for every sample
y_pred_baseline = [y_mean] * len(y_train)

print("Mean apt price:", y_mean)
print("Baseline MAE:", mean_absolute_error(y_train, y_pred_baseline))


In [ ]:
model = make_pipeline(
    OneHotEncoder(use_cat_names=True),
    SimpleImputer(),
    Ridge()
)
model.fit(X_train,y_train)


In [ ]:
# Check your work
check_is_fitted(model[-1])


In [ ]:
VimeoVideo("656849505", h="f153a4f005", width=600)


In [ ]:
y_pred_training = model.predict(X_train)
print("Training MAE:", mean_absolute_error(y_train,y_pred_training))


In [ ]:
X_test = pd.read_csv("data/buenos-aires-test-features.csv")[feature_matrix]
y_pred_test = pd.Series(model.predict(X_test))
y_pred_test.head()


In [ ]:
VimeoVideo("656849254", h="e6faad47ca", width=600)


In [ ]:
def make_prediction(area, lat, lon, neighborhood):
    data = {
        "surface_covered_in_m2": area,
        "lat": lat,
        "lon": lon,
        "neighborhood": neighborhood
    }
    df = pd.DataFrame(data, index=[0])
    prediction = model.predict(df).round(2)
    return f"Predicted apartment price: ${prediction}"


In [ ]:
make_prediction(110, -34.60, -58.46, "Villa Crespo")


In [ ]:
VimeoVideo("656848911", h="7939dcd479", width=600)


In [ ]:
interact(
    make_prediction,
    area=IntSlider(
        min=X_train["surface_covered_in_m2"].min(),
        max=X_train["surface_covered_in_m2"].max(),
        value=X_train["surface_covered_in_m2"].mean(),
    ),
    lat=FloatSlider(
        min=X_train["lat"].min(),
        max=X_train["lat"].max(),
        step=0.01,
        value=X_train["lat"].mean(),
    ),
    lon=FloatSlider(
        min=X_train["lon"].min(),
        max=X_train["lon"].max(),
        step=0.01,
        value=X_train["lon"].mean(),
    ),
    neighborhood=Dropdown(options=sorted(X_train["neighborhood"].unique())),
);
